# 🗂️ Notebook 2: Reddit — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/reddit
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data model

```sql
users        (id, name)
subreddits   (id, name, created_at)
memberships  (user_id, subreddit_id)
posts        (id, subreddit_id, author_id, title, body|url, created_at,
              upvotes, downvotes, hot_score)     -- hot_score denormalized
comments     (id, post_id, parent_id, author_id, body, created_at)
votes        (user_id, post_or_comment_id, kind, value ∈ {-1,+1})
```

### Denormalization of scores
Reading a post should not aggregate from `votes`. We **denormalize** the counts and the
computed hot score on the post row, updated by the ranking job (or incrementally on vote).

## Key APIs

```http
POST /r/{name}/posts        create post
POST /posts/{id}/vote       { dir: +1 | -1 | 0 }
GET  /r/{name}?sort=hot     feed
GET  /r/all?sort=hot        cross-sub feed
GET  /posts/{id}/comments?limit=...&more_children=...
```


In [ ]:
from pydantic import BaseModel
from typing import Literal
class VoteRequest(BaseModel):
    dir: Literal[-1, 0, 1]
print(VoteRequest(dir=1).model_dump_json())
